**Task 1: Order Flow Imbalances (OFI)**


In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

df = pd.read_csv('first_25000_rows.csv', parse_dates=['ts_event'])
df = df.sort_values(['symbol', 'ts_event']).reset_index(drop=True)

# Compute event-level OFI for 10 book levels (00→ofi_1 … 09→ofi_10)
def compute_event_ofi(df):
    df = df.copy()
    for m in range(10):
        lvl = f'{m:02d}'
        px_b, sz_b = df[f'bid_px_{lvl}'], df[f'bid_sz_{lvl}']
        px_a, sz_a = df[f'ask_px_{lvl}'], df[f'ask_sz_{lvl}']
        dpx_b, dsz_b = px_b.diff().fillna(0), sz_b.diff().fillna(0)
        dpx_a, dsz_a = px_a.diff().fillna(0), sz_a.diff().fillna(0)
        ofb = np.where(dpx_b>0, sz_b, np.where(dpx_b==0, dsz_b, -sz_b))
        ofa = np.where(dpx_a>0, -sz_a, np.where(dpx_a==0, dsz_a, sz_a))
        df[f'ofi_{m+1}'] = ofb - ofa
    return df[['symbol', 'ts_event'] + [f'ofi_{i}' for i in range(1,11)]]

ofi_events = compute_event_ofi(df)

Best-Level OFI

In [2]:
# sum of ofi_1 per 1 min per symbol
best_ofi = (
    ofi_events
    .set_index('ts_event')
    .groupby('symbol')['ofi_1']
    .resample('min')
    .sum()
    .rename('best_ofi')
    .reset_index()
)
best_ofi

,symbol,ts_event,best_ofi
0,AAPL,2024-10-21 11:54:00+00:00,-531.0
1,AAPL,2024-10-21 11:55:00+00:00,-1217.0
2,AAPL,2024-10-21 11:56:00+00:00,334.0
3,AAPL,2024-10-21 11:57:00+00:00,960.0
4,AAPL,2024-10-21 11:58:00+00:00,615.0
...,...,...,...
66,AAPL,2024-10-21 13:00:00+00:00,-1007.0
67,AAPL,2024-10-21 13:01:00+00:00,-1137.0
68,AAPL,2024-10-21 13:02:00+00:00,-6296.0
69,AAPL,2024-10-21 13:03:00+00:00,171.0


Multi-Level OFI

In [3]:
# sums of ofi_1…ofi_10 per 1 min
multi_ofi = (
    ofi_events
    .set_index('ts_event')
    .groupby('symbol')[[f'ofi_{i}' for i in range(1,11)]]
    .resample('min')
    .sum()
)
multi_ofi

ofi_1   ofi_2   ofi_3   ofi_4   ofi_5  \
symbol ts_event                                                            
AAPL   2024-10-21 11:54:00+00:00  -531.0   346.0   542.0  -156.0   -61.0   
       2024-10-21 11:55:00+00:00 -1217.0  1767.0  1821.0  -236.0  -380.0   
       2024-10-21 11:56:00+00:00   334.0  -573.0  1897.0  -585.0   652.0   
       2024-10-21 11:57:00+00:00   960.0   653.0   335.0   205.0 -1765.0   
       2024-10-21 11:58:00+00:00   615.0    18.0  1047.0   590.0   -41.0   
...                                  ...     ...     ...     ...     ...   
       2024-10-21 13:00:00+00:00 -1007.0   795.0  -343.0 -2598.0  2395.0   
       2024-10-21 13:01:00+00:00 -1137.0 -2111.0 -4484.0  2006.0  -154.0   
       2024-10-21 13:02:00+00:00 -6296.0 -4791.0 -5286.0  -328.0  -590.0   
       2024-10-21 13:03:00+00:00   171.0 -1110.0  1402.0  1464.0  1169.0   
       2024-10-21 13:04:00+00:00  -196.0 -1011.0  -468.0   135.0  -610.0   

                                   ofi_6   ofi_7   ofi_8   ofi_9  ofi_10  
symbol ts_event                                                           
AAPL   2024-10-21 11:54:00+00:00   580.0    44.0  -247.0   677.0   590.0  
       2024-10-21 11:55:00+00:00   738.0  -789.0   161.0  1388.0  -580.0  
       2024-10-21 11:56:00+00:00  -272.0   -64.0   850.0  -144.0   253.0  
       2024-10-21 11:57:00+00:00   986.0  -504.0  -379.0  1667.0  -235.0  
       2024-10-21 11:58:00+00:00 -1183.0  1191.0  -111.0   567.0  2455.0  
...                                  ...     ...     ...     ...     ...  
       2024-10-21 13:00:00+00:00 -1808.0  2360.0   921.0 -1079.0   629.0  
       2024-10-21 13:01:00+00:00    85.0  1215.0   816.0  -833.0  2125.0  
       2024-10-21 13:02:00+00:00 -1420.0 -1503.0 -1802.0  3148.0 -2123.0  
       2024-10-21 13:03:00+00:00  1332.0  2463.0  -866.0   187.0  2530.0  
       2024-10-21 13:04:00+00:00  -508.0   124.0  -985.0  -553.0  -155.0  

[71 rows x 10 columns]

Integrated OFI

In [4]:
# first PCA component of the 10-level OFIs, L1-normalized
def integrate_ofi(df_block):
    pca = PCA(n_components=1)
    w = pca.fit(df_block).components_[0]
    return df_block.values.dot(w) / np.abs(w).sum()

int_rows = []
for sym, grp in multi_ofi.groupby('symbol'):
    data = grp.droplevel(0)
    ints = integrate_ofi(data)
    tmp = pd.DataFrame({
      'symbol': sym,
      'ts_event': data.index,
      'integrated_ofi': ints
    })
    int_rows.append(tmp)

integrated_ofi = pd.concat(int_rows).reset_index(drop=True)
integrated_ofi 

,symbol,ts_event,integrated_ofi
0,AAPL,2024-10-21 11:54:00+00:00,-102.274901
1,AAPL,2024-10-21 11:55:00+00:00,-248.491093
2,AAPL,2024-10-21 11:56:00+00:00,-829.879615
3,AAPL,2024-10-21 11:57:00+00:00,585.957694
4,AAPL,2024-10-21 11:58:00+00:00,-306.183605
...,...,...,...
66,AAPL,2024-10-21 13:00:00+00:00,-1560.614489
67,AAPL,2024-10-21 13:01:00+00:00,1603.988434
68,AAPL,2024-10-21 13:02:00+00:00,1099.444606
69,AAPL,2024-10-21 13:03:00+00:00,-154.479218


Cross-Asset OFI

In [5]:
# pivot best_ofi into wide format (timestamps × symbols)
# I found that only one symbol "AAPL" is present in the dataset, so this “cross-asset” matrix will trivially consist of a single column equal to that symbol’s Best-Level OFI.
cross_asset_ofi = (
    best_ofi
    .pivot(index='ts_event', columns='symbol', values='best_ofi')
    .fillna(0)
)
cross_asset_ofi

symbol,AAPL
ts_event,
2024-10-21 11:54:00+00:00,-531.0
2024-10-21 11:55:00+00:00,-1217.0
2024-10-21 11:56:00+00:00,334.0
2024-10-21 11:57:00+00:00,960.0
2024-10-21 11:58:00+00:00,615.0
...,...
2024-10-21 13:00:00+00:00,-1007.0
2024-10-21 13:01:00+00:00,-1137.0
2024-10-21 13:02:00+00:00,-6296.0
